## Local Compression

### Vanilla Layer Compression with SVD

CNN Weights start as the tensor W (filters, channels, k_h, k_w), and are first transformed into a 2D tensor of shape $f \times ck_hk_w$. 

Applying SVD on $W$ yields

$$
\begin{align*}
W &= U\Sigma V^T \\
&= \sum_{i=1}^{f}\sigma_i u_i v_i^T,
\end{align*}
$$

We know that each $\sigma_i u_i v_i^T$ is a rank-1 matrix, and so we can construct a rank-$j$ approximation of $W$ by taking the first $j$ of these terms:
$$
\begin{align*}
W &\approx \sum_{i=1}^{j}\sigma_i u_i v_i^T \\
&\approx U_j\Sigma_j V^T_j \\
&\approx U_j(\Sigma_j V^T_j)
\end{align*}
$$

where (letting $d = ck_hk_w$)
- $U_j \in \mathbb{R}^{f \times j}$
- $\Sigma_j \in \mathbb{R}^{j \times j}$
- $V^T_j \in \mathbb{R}^{j \times d}$

Then, $$W \approx U V \quad \text{where} \quad U = U_j, V = \Sigma_j V^T_j$$

Using this, we can separate the linear $f \times d$ layer into two linear layers of $U \in \mathbb{R}^{f \times j}$ and $V \in \mathbb{R}^{j \times d}$. 

When we map these back into convolutions, the result is that we multiply we get a $f \times j \times 1 \times 1$ convolution and a $j \times c \times k_h \times k_w$ convolution.

Originally, the convolution had $fd$ parameters, but we've now reduced it to $fj+ jd$, or $j(f + d)$ parameters

![vanilla_svd.png](images/vanilla_svd.png)


### Channel Slicing


One global rank for each CNN layer is too restrictive, a better representation of the layer would be a local rank approximation of different chunks of the layer.

This gets you closer to a rank-$kj$ approximation without having the pay the full rank-$kj$ parameter cost.
$kj(f+d)$


Thus, we split the input channels into $k$ groups, each of size $c/k$

So $$W = [W_1 \mid W_2 \mid ... \mid W_k]$$

And instead of decomposing $W$, we decompose each $W_i$, $i \in [k]$
$$W_i \approx U_i V_i$$
Where $$U_i \in \mathbb{R}^{f \times j}, \quad V_i \in \mathbb{R}^{j \times c_i k_h k_w}$$

So $$W \approx [U_1V_1 \mid U_2V_2 \mid ... \mid U_k V_k]$$

Each $V_i$ becomes a $j \times c_i \times k_h \times k_w$ convolution, and these $k$ convolutions are stacked in parallel 

Each $U_i$ is reshaped to $f \times j \times 1 \times 1$ and they are channel-stacked to form a $f \times kj \times 1 \times 1$ convolution.

2d feature map, feature is just a pattern. talk about how the input moves thorugh cnn


![convolution_to_matrix_multiplication.png](images/convolution_to_matrix_multiplication.png)


### Why split on channels rather than filters?


Splitting on channels is computationally more efficient than on filters because the expensive computation is on the input side $(ck_hk_w)$,

Splitting on channels:
- Each vertical block acts on only a subspace of the input space, and then maps onto a shared output space.

Split on filters:
- Each horizontal block acts on the entire input space, and instead the output space is split.


We have a way to compress and reduce parameters, but 

What's different about weight compression in neural networks is that you don't want to focus on the resolution loss of the weight tensors themselves, but rather the loss in the predictions of the network.

Hyperparameters: Number of subspaces to split each matrix $k^l$ and rank $j^l$ for each layer
Minimize: The drop in the accuracy of the network.

It is very computationally extensive to bindly brute force the solution, so the paper uses the Eckhart-Young Mirsky Theorem to find the bounds of the solution.

- compare original loss with surrogate
- Use simple brute force on j and k to show how long it takes 
- compare channel slicing vs no channel slicing

linear algebra to include
- svd, rank j
- eckart
- theorem 1
- minimax


https://arxiv.org/pdf/2107.11442

talk abt process rather than results
storytelling